# Module 09: High-Performance Persistence & Parallel Computing with Joblib
## Notebook 03: Transparent Function and Pipeline Caching with `joblib.Memory`

Data science workflows involve repetitive, compute-intensive operations: loading raw data, tokenizing text, computing PCA projections, and training feature scalers. Rerunning these computations repeatedly wastes hours of CPU time.
**`joblib.Memory`** provides transparent disk-based memoization for Python functions, intelligently hashing complex arguments (including NumPy arrays and DataFrames).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Initialize and configure a `joblib.Memory` disk caching environment.
2. Decorate expensive Python functions with **`@memory.cache`**.
3. Understand how Joblib computes deterministic cryptographic hashes of large NumPy arrays and Pandas DataFrames.
4. **Advanced:** Integrate `joblib.Memory` directly into **Scikit-Learn Pipelines** to cache intermediate feature extraction steps during `GridSearchCV`.
5. **Advanced:** Inspect cache directory metadata, diagnose cache hits vs. misses, and execute programmatic cache pruning.

In [ ]:
import os
import shutil
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

cache_dir = "joblib_cache_store"
memory = joblib.Memory(location=cache_dir, verbose=0)
print(f"Configured Joblib Memory Cache at: {os.path.abspath(cache_dir)}")

### 1. Transparent Function Memoization
When a decorated `@memory.cache` function is called:
1. Joblib hashes the function bytecode and all input arguments.
2. If the hash exists in the cache directory, Joblib loads the precomputed output directly from disk (**Cache Hit**, latency < 1 ms).
3. If the hash is new, Joblib executes the function, writes the output to disk, and returns the result (**Cache Miss**).

In [ ]:
# Define an expensive numerical transformation function
@memory.cache
def expensive_feature_synthesis(data_matrix, power=2, sleep_sec=1.0):
    time.sleep(sleep_sec) # Simulate intensive computation
    return np.power(data_matrix, power) + np.sin(data_matrix)

# Create input matrix
test_data = np.random.uniform(0, 10, size=(1000, 50))

# 1. First execution (CACHE MISS: Takes ~1.0 second)
t0 = time.time()
res1 = expensive_feature_synthesis(test_data, power=2, sleep_sec=1.0)
duration_miss = (time.time() - t0) * 1000

# 2. Second execution with identical arguments (CACHE HIT: Takes < 5 ms)
t0 = time.time()
res2 = expensive_feature_synthesis(test_data, power=2, sleep_sec=1.0)
duration_hit = (time.time() - t0) * 1000

print(f"First Call (Cache Miss): {duration_miss:.2f} ms")
print(f"Second Call (Cache Hit):  {duration_hit:.2f} ms")
print(f"Speedup Factor: {duration_miss / max(duration_hit, 1e-5):.1f}x faster!")

np.testing.assert_array_equal(res1, res2)

### 2. Complex Application 1: Caching Scikit-Learn Pipelines during Hyperparameter Search
When tuning classifier hyperparameters in a pipeline `[Scaler -> PCA -> Classifier]`:
- Standard `GridSearchCV` recomputes `StandardScaler.fit_transform()` and `PCA.fit_transform()` for **every single hyperparameter combination**!
- By passing `memory=cache_dir` to `Pipeline`, Scikit-Learn caches the intermediate outputs of `Scaler` and `PCA`.
- The expensive PCA transformation runs **only once per cross-validation fold**, accelerating grid search by up to $10\times$!

In [ ]:
# Synthesize high-dimensional classification problem
X_heavy = np.random.randn(800, 60)
y_heavy = (np.sum(X_heavy[:, :5], axis=1) > 0).astype(int)

# 1. Pipeline WITH Joblib Memory Caching
pipeline_cached = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=15)),
    ("svm", SVC())
], memory=cache_dir)

# Define hyperparameter grid for the SVM classifier only
param_grid = {
    "svm__C": [0.1, 1.0, 10.0],
    "svm__gamma": ["scale", "auto"]
}

# Run Grid Search across 3 folds (6 parameter combinations x 3 folds = 18 fits)
grid_search = GridSearchCV(pipeline_cached, param_grid, cv=3, n_jobs=1)

t0 = time.time()
grid_search.fit(X_heavy, y_heavy)
cached_grid_time = time.time() - t0

print(f"Cached Grid Search Time (18 fits): {cached_grid_time:.2f} seconds")
print(f"Optimal Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.3f}")

### 3. Complex Application 2: Cache Inspection & Programmatic Cleanup
In production systems, disk caches can grow indefinitely. `joblib.Memory` provides tools to inspect cache directory sizes and clear obsolete caches.

In [ ]:
# Inspect total size of cache store on disk
def get_dir_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

cache_bytes = get_dir_size(cache_dir)
print(f"Total Cache Store Footprint: {cache_bytes / 1024:.2f} KB")

# Programmatically clear cache
memory.clear(warn=False)
print("SUCCESS: Memory cache cleared successfully!")

# Clean up physical directory
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)